# Phase 2a 얼굴·안경 검수
`extract_bbox.py`가 저장한 주석에서 클래스별 최대 5장을 확인한다. 초록은 선택한 얼굴, 청록은 다른 얼굴 후보, 빨강은 안경 bbox다. 얼굴 ID와 착용 점수를 원본 위에 표시한다.

In [ ]:
from pathlib import Path

cwd = Path.cwd()
root = next(
    (path for path in (cwd, *cwd.parents, Path.home() / "JeongGeon/BUTAMI")
     if (path / "code/phase2a_extract/bbox/extract_bbox.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("BUTAMI 저장소나 단계별 코드 폴더에서 노트북을 여세요.")

import json
import random
from collections import defaultdict

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps

data = json.loads((root / "data/annotations/face_glasses.json").read_text(encoding="utf-8"))
annotations = defaultdict(list)
for ann in data["annotations"]:
    annotations[ann["image_id"]].append(ann)

rng = random.Random(42)
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for ax in axes.flat:
    ax.axis("off")

for row, label in enumerate(("no_glasses", "regular_glasses", "smart_glasses")):
    images = [info for info in data["images"] if info["file_name"].split("/")[0] == label]
    for ax, info in zip(axes[row], rng.sample(images, min(5, len(images)))):
        with Image.open(root / "data/images" / info["file_name"]) as original:
            ax.imshow(ImageOps.exif_transpose(original).convert("RGB"))
        for ann in annotations[info["id"]]:
            x, y, w, h = ann["bbox"]
            if ann["category_id"] == 1:
                color = "lime" if ann["id"] == info.get("selected_face_id") else "cyan"
                label_text = f"face {ann['id']} | score={ann['score']:.3f}"
                probability = ann.get("glasses_probability")
                if probability is not None:
                    label_text += f" | glasses={probability:.3f}"
            elif ann["category_id"] == 2:
                color = "red"
                label_text = f"glasses | face {ann.get('face_id', '?')}"
            else:
                continue
            ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
            ax.text(x, y, label_text, color=color, va="top", fontsize=8,
                    bbox=dict(facecolor="black", alpha=0.65, edgecolor="none"))
        ax.set_title(info["file_name"], fontsize=9)

fig.tight_layout()
plt.show()